# Notebook 4: Lick Decoder
Logistic regression on PCA-reduced population activity (session 1044385384).

**Before running:** add your GitHub token as a Colab Secret:
1. Click the 🔑 key icon in the left sidebar
2. Add a secret named `GITHUB_TOKEN` with your Personal Access Token as the value
3. Toggle it on for this notebook

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import os, sys

REPO = "/content/allen-neuropixels-decoder"

# Clone repo if not already present; pull latest if it is
if not os.path.exists(REPO):
    os.system(f"git clone https://github.com/jadenbalajadia/allen-neuropixels-decoder.git {REPO}")
else:
    os.system(f"git -C {REPO} pull")

sys.path.append(REPO)

from google.colab import drive, userdata
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/MyDrive/lickingpcaoutputs")
print("Setup complete. DATA_DIR:", DATA_DIR)

In [ ]:
# ── Cell 2: Load data ──────────────────────────────────────────────────────
from src.preprocessing import load_arrays

X_raw, y, labels = load_arrays(DATA_DIR)
print(f"X_raw: {X_raw.shape}  y: {y.shape}")
print(f"Class balance — lick bins: {y.mean()*100:.1f}%  no-lick bins: {(1-y.mean())*100:.1f}%")

SIGMA_BINS   = 2   # ~100 ms at 50 ms / bin — smoothing happens inside each CV fold
N_COMPONENTS = 20

In [ ]:
# ── Cell 2b: Lick rate over time ───────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BIN_SIZE = 0.05          # seconds per bin (20 Hz)
bins_per_min = int(60 / BIN_SIZE)   # 1200 bins = 1 minute
n_minutes = len(y) // bins_per_min

lick_rate = np.array([
    y[i * bins_per_min : (i + 1) * bins_per_min].sum()
    for i in range(n_minutes)
])

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(n_minutes), lick_rate, color="steelblue", width=0.85, alpha=0.8)
ax.axvline(59.5, color="crimson", linestyle="--", linewidth=1.2,
           label="engagement ends (min 60)")
ax.set_xlabel("Time (minutes)")
ax.set_ylabel("Licks per minute")
ax.set_title("Lick Rate Over Time — Session 1044385384")
ax.legend(fontsize=9)
plt.tight_layout()

FIGURES_DIR = Path(REPO) / "figures"
FIGURES_DIR.mkdir(exist_ok=True)
fig.savefig(FIGURES_DIR / "lick_rate_over_time.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR / 'lick_rate_over_time.png'}")

In [ ]:
# ── Cell 3b: Blocked (contiguous-time) CV ──────────────────────────────────
# StratifiedKFold(shuffle=True) scatters 50 ms bins randomly; neighboring bins
# are highly correlated at 20 Hz, so a test bin can be near-duplicated by a
# training bin just milliseconds away, slightly inflating AUC.
# Blocked CV assigns each contiguous session chunk to one fold for genuinely
# independent train/test windows.
#
# NOTE: lick events in this session are concentrated in the first ~37% of the
# recording (first 60 of 162 minutes). Folds that fall entirely outside that
# window are skipped (no positive labels → AUC is undefined). Only folds with
# ≥1 lick are scored.
from src.decoder import cross_validate_blocked

aucs_blocked = cross_validate_blocked(
    X_raw, y, sigma_bins=SIGMA_BINS, n_components=N_COMPONENTS, n_splits=5
)
print(f"\nPer-fold: {[f'{a:.4f}' for a in aucs_blocked]}")

import numpy as np
print(f"\n{'═'*50}")
print(f"  Shuffled k-fold AUC : {np.array(aucs).mean():.3f} ± {np.array(aucs).std():.3f}  (n=5 folds)")
print(f"  Blocked CV AUC      : {np.array(aucs_blocked).mean():.3f} ± {np.array(aucs_blocked).std():.3f}  ({len(aucs_blocked)}/5 folds with lick events)")
print(f"  Gap reflects temporal autocorrelation between adjacent 50 ms bins.")
print(f"{'═'*50}")

In [ ]:
# ── Cell 3b: Blocked (contiguous-time) CV ──────────────────────────────────
# StratifiedKFold(shuffle=True) scatters 50 ms bins randomly; neighboring bins
# are highly correlated at 20 Hz, so a test bin can be near-duplicated by a
# training bin just milliseconds away, slightly inflating AUC.
# Blocked CV assigns each contiguous session chunk to one fold for genuinely
# independent train/test windows.
#
# NOTE: lick events in this session are concentrated in the first ~40% of the
# recording. Folds that fall entirely outside that window are skipped
# (no positive labels → AUC is undefined). Only folds with ≥1 lick are scored.
from src.decoder import cross_validate_blocked

aucs_blocked = cross_validate_blocked(
    X_raw, y, sigma_bins=SIGMA_BINS, n_components=N_COMPONENTS, n_splits=5
)
print(f"\nPer-fold: {[f'{a:.4f}' for a in aucs_blocked]}")

import numpy as np
print(f"\n{'═'*50}")
print(f"  Shuffled k-fold AUC : {np.array(aucs).mean():.3f} ± {np.array(aucs).std():.3f}  (n=5 folds)")
print(f"  Blocked CV AUC      : {np.array(aucs_blocked).mean():.3f} ± {np.array(aucs_blocked).std():.3f}  ({len(aucs_blocked)}/5 folds with lick events)")
print(f"  Gap reflects temporal autocorrelation between adjacent 50 ms bins.")
print(f"{'═'*50}")

In [ ]:
# ── Cell 4: ROC curve ──────────────────────────────────────────────────────
from src.decoder import plot_roc

os.makedirs(f"{REPO}/figures", exist_ok=True)

fig, mean_auc = plot_roc(X_raw, y, sigma_bins=SIGMA_BINS, n_components=N_COMPONENTS, n_splits=5)
fig.savefig(f"{REPO}/figures/roc_corrected.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Mean AUC: {mean_auc:.3f}")

In [ ]:
# ── Cell 5: Push figure to GitHub ──────────────────────────────────────────
# Reads your token from Colab Secrets — never hardcode it in the notebook.
import subprocess

token = userdata.get("GITHUB_TOKEN")  # set this in the 🔑 Secrets panel

def git(cmd):
    result = subprocess.run(
        ["git", "-C", REPO] + cmd,
        capture_output=True, text=True
    )
    if result.stdout: print(result.stdout.strip())
    if result.stderr: print(result.stderr.strip())

git(["config", "user.email", "jadenbalajadia4@gmail.com"])
git(["config", "user.name",  "jadenbalajadia"])
git(["add",    "figures/roc_corrected.png"])
git(["commit", "-m", "Add corrected ROC curve (AUC 0.955)"])
git(["push",   f"https://{token}@github.com/jadenbalajadia/allen-neuropixels-decoder.git"])

print("Done — check github.com/jadenbalajadia/allen-neuropixels-decoder")